# Extraction, Transform, Validate, Load
**Oleh:** Yazitzy

Perbaikan dari versi asli: menambahkan step **Validate** sebelum Load
(sebelumnya notebook ini langsung lompat dari Transform ke Load, padahal
halaman 6 PDF portofolio menjelaskan Validation Gate sebagai bagian
penting dari pipeline).

In [1]:
import pandas as pd
import numpy as np

# === EXTRACT ===
orders = pd.read_csv('raw_orders.csv')
products = pd.read_csv('raw_products.csv')

print("=== ORDERS INFO ===")
print(f"Jumlah baris: {len(orders)}")
print(f"Kolom: {list(orders.columns)}")
print(orders.info())
print()
print("=== CEK MASALAH ===")
print(f"Duplikasi: {orders.duplicated().sum()}")
print(f"Missing values:")
print(orders.isnull().sum())
print(f"\nHarga negatif: {(orders['total_harga'] < 0).sum()}")
print(f"\nChannel unik: {orders['channel'].unique()}")
print(f"Kota unik: {orders['kota'].unique()}")

=== ORDERS INFO ===
Jumlah baris: 130
Kolom: ['order_id', 'product_id', 'product_name', 'kategori', 'quantity', 'total_harga', 'tanggal_order', 'kota', 'channel', 'status', 'customer_email']
<class 'pandas.DataFrame'>
RangeIndex: 130 entries, 0 to 129
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   order_id        130 non-null    str    
 1   product_id      130 non-null    str    
 2   product_name    130 non-null    str    
 3   kategori        130 non-null    str    
 4   quantity        130 non-null    int64  
 5   total_harga     125 non-null    float64
 6   tanggal_order   130 non-null    str    
 7   kota            130 non-null    str    
 8   channel         130 non-null    str    
 9   status          130 non-null    str    
 10  customer_email  110 non-null    str    
dtypes: float64(1), int64(1), str(9)
memory usage: 11.3 KB
None

=== CEK MASALAH ===
Duplikasi: 10
Missing values:
order_id       

In [2]:
# === TRANSFORM ===

# 2a. Hapus duplikasi
print(f"Sebelum: {len(orders)} baris")
orders = orders.drop_duplicates()
print(f"Setelah hapus duplikat: {len(orders)} baris")

# 2b. Hapus baris dengan harga negatif (data error)
orders = orders[orders['total_harga'] >= 0]
print(f"Setelah hapus harga negatif: {len(orders)} baris")

# 2c. Isi missing values
orders['customer_email'] = orders['customer_email'].fillna('unknown@placeholder.com')
median_harga = orders['total_harga'].median()
orders['total_harga'] = orders['total_harga'].fillna(median_harga)
print(f"Missing values setelah fillna: {orders.isnull().sum().sum()}")

# 2d. Standarkan format tanggal
# PENTING: kolom ini punya 3 format (ISO, slash, teks), jadi butuh
# format='mixed' supaya pandas menebak format per baris. dayfirst=True
# saja (tanpa format='mixed') sempat dipakai di versi sebelumnya dan
# ternyata error di pandas versi ini -- kombinasi keduanya yang robust.
orders['tanggal_order'] = pd.to_datetime(
    orders['tanggal_order'],
    format='mixed',
    dayfirst=True
)
print(f"Tipe tanggal: {orders['tanggal_order'].dtype}")

# 2e. Standarkan teks (lowercase lalu title case)
orders['kota'] = orders['kota'].str.strip().str.title()
orders['channel'] = orders['channel'].str.strip().str.lower().str.replace(' ', '_')
print(f"Channel setelah standarisasi: {orders['channel'].unique()}")
print(f"Kota setelah standarisasi: {orders['kota'].unique()}")

# 2f. Buat kolom baru: bulan dan kategori harga
orders['bulan'] = orders['tanggal_order'].dt.month_name()
orders['kategori_harga'] = np.where(
    orders['total_harga'] < 500000, 'kecil',
    np.where(orders['total_harga'] <= 2000000, 'sedang', 'besar')
)

print(f"\nDistribusi kategori harga:")
print(orders['kategori_harga'].value_counts())

Sebelum: 130 baris
Setelah hapus duplikat: 120 baris
Setelah hapus harga negatif: 110 baris
Missing values setelah fillna: 0
Tipe tanggal: datetime64[us]
Channel setelah standarisasi: <StringArray>
['website', 'mobile_app', 'marketplace']
Length: 3, dtype: str
Kota setelah standarisasi: <StringArray>
[      'Solo',    'Bandung',   'Makassar',     'Malang',   'Surabaya',
 'Yogyakarta',      'Medan',   'Semarang',   'Denpasar',    'Jakarta']
Length: 10, dtype: str

Distribusi kategori harga:
kategori_harga
sedang    51
besar     49
kecil     10
Name: count, dtype: int64


## Validate
Quality gate sebelum Load -- kalau ada check yang gagal, pipeline berhenti di sini dan data kotor tidak pernah disimpan.

In [3]:
# === VALIDATE ===
checks = {
    'Tidak ada duplikat': orders.duplicated().sum() == 0,
    'Tidak ada missing value': orders.isnull().sum().sum() == 0,
    'Tidak ada harga negatif': (orders['total_harga'] < 0).sum() == 0,
    'Tanggal tipe datetime': str(orders['tanggal_order'].dtype).startswith('datetime'),
    'Channel konsisten': len(orders['channel'].unique()) <= 3,
}

print("=== VALIDASI DATA BERSIH ===")
for check, passed in checks.items():
    print(f"  {'✅' if passed else '❌'} {check}")

all_passed = all(checks.values())
if not all_passed:
    raise ValueError("Validasi gagal, pipeline dihentikan sebelum Load!")
print(f"\nHasil: {'SEMUA LOLOS' if all_passed else 'ADA YANG GAGAL'}")

=== VALIDASI DATA BERSIH ===
  ✅ Tidak ada duplikat
  ✅ Tidak ada missing value
  ✅ Tidak ada harga negatif
  ✅ Tanggal tipe datetime
  ✅ Channel konsisten

Hasil: SEMUA LOLOS


In [4]:
# === LOAD ===
# Di dunia nyata: load ke BigQuery / data warehouse
# Di exercise ini: simpan ke CSV bersih

orders_clean = orders[[
    'order_id', 'product_id', 'product_name', 'kategori',
    'quantity', 'total_harga', 'tanggal_order', 'kota',
    'channel', 'status', 'customer_email', 'bulan', 'kategori_harga'
]]

orders_clean.to_csv('orders_clean.csv', index=False)
print(f"Data bersih disimpan: orders_clean.csv ({len(orders_clean)} baris)")

# Buat summary report
summary = orders_clean.groupby('kategori').agg(
    total_orders=('order_id', 'count'),
    total_revenue=('total_harga', 'sum'),
    avg_revenue=('total_harga', 'mean')
).round(0)

print("\n=== SUMMARY PER KATEGORI ===")
print(summary)

summary.to_csv('summary_report.csv')
print("\nSummary disimpan: summary_report.csv")

Data bersih disimpan: orders_clean.csv (110 baris)

=== SUMMARY PER KATEGORI ===
            total_orders  total_revenue  avg_revenue
kategori                                            
Elektronik            81    435180000.0    5372593.0
Furniture             29    127350000.0    4391379.0

Summary disimpan: summary_report.csv
